# 01 — Bronze Layer Ingestion

**Purpose:** Ingest all four source CSV files into raw Delta tables in the Bronze layer.

**What this notebook does:**
- Accepts `pipeline_run_date` as a parameter (passed from the Fabric pipeline)
- Reads each CSV from the Lakehouse `Files/raw/` section
- Adds four metadata columns: `_source_file`, `_source_system`, `_ingested_at`, `_ingestion_date`
- Appends each file as a partitioned Delta table — one partition per run date
- No transformations, no business logic, no joins — Bronze is a pure audit layer

**Output tables:**
- `bronze.hubspot_leads`
- `bronze.dynamics_applications`
- `bronze.paradigm_enrolments`
- `bronze.canvas_attendance`

**Partition strategy:** `(_source_system, _ingestion_date)`
Every pipeline run appends a new date partition — full history is preserved.
Silver reads only the partition matching `pipeline_run_date` — no duplicates on reruns.

> **Design principle:** Bronze is the audit layer. Never overwrite it.
> If Silver or Gold breaks, replay from any Bronze partition by date.
> Each run is independently queryable: `WHERE _ingestion_date = '2024-03-01'`

## Parameters

When run via the Fabric pipeline, `pipeline_run_date` is injected automatically.
When run manually, it defaults to today's date.

This parameter is the single clock that drives the entire stack:
- Bronze partitions by this date
- Silver reads only this date's Bronze partition
- All notebooks stay in sync on what 'today' means

In [ ]:
# Default value — overridden by Fabric pipeline at runtime
# Tag this cell as 'parameters' in Fabric so the pipeline can inject the value
pipeline_run_date = "2024-01-15"  # YYYY-MM-DD

## Setup

Import libraries and define the ingestion helper function.
The helper encapsulates the four-step Bronze pattern:
read → add metadata → append → partition by (source_system, ingestion_date).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# Base path to CSV files uploaded to Lakehouse Files section
FILES_BASE_PATH = "Files/raw"

# Cast pipeline_run_date string to a proper date for use as partition value
run_date = pipeline_run_date  # e.g. '2024-01-15'

print(f"Pipeline run date: {run_date}")
print(f"Files base path:   {FILES_BASE_PATH}")
print()


def ingest_to_bronze(filename: str, source_system: str, table_name: str) -> None:
    """
    Reads a CSV from Lakehouse Files/raw/, adds metadata columns,
    and appends as a partitioned Delta table in the bronze schema.

    Partition key: (_source_system, _ingestion_date)
    Mode: append — each run adds a new date partition, never overwrites.

    Args:
        filename:      CSV filename e.g. 'hubspot_leads.csv'
        source_system: Source system label e.g. 'HUBSPOT'
        table_name:    Target Delta table name e.g. 'hubspot_leads'
    """
    source_path = f"{FILES_BASE_PATH}/{filename}"
    target_table = f"bronze.{table_name}"

    print(f"[{source_system}] Reading: {source_path}")

    # Read CSV — infer schema, use header row, treat empty strings as NULL
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("emptyValue", None)
        .csv(source_path)
    )

    row_count = df.count()
    print(f"[{source_system}] Rows read:  {row_count}")
    print(f"[{source_system}] Columns:    {df.columns}")

    # Add four metadata columns — this is the only change Bronze makes to the data
    # _ingestion_date is derived from pipeline_run_date, not current_date()
    # This ensures reruns on a different calendar day still write to the correct partition
    df = (
        df
        .withColumn("_source_file",    F.lit(filename))
        .withColumn("_source_system",  F.lit(source_system))
        .withColumn("_ingested_at",    F.current_timestamp())
        .withColumn("_ingestion_date", F.lit(run_date).cast(DateType()))
    )

    # Check if this partition already exists — warn but do not block
    # In production you would raise an error to prevent duplicate ingestion
    # For demo purposes we allow reruns to overwrite the same date partition
    try:
        existing = spark.table(target_table)
        already_loaded = (
            existing
            .filter(
                (F.col("_source_system") == source_system) &
                (F.col("_ingestion_date") == run_date)
            )
            .count()
        )
        if already_loaded > 0:
            print(f"[{source_system}] WARNING: partition {run_date} already has {already_loaded} rows — appending again")
    except Exception:
        pass  # Table does not exist yet — first run

    # Append to Delta table — partitioned by source system and ingestion date
    # This creates a new folder for each (source_system, ingestion_date) combination
    (
        df.write
        .format("delta")
        .mode("append")
        .partitionBy("_source_system", "_ingestion_date")
        .saveAsTable(target_table)
    )

    print(f"[{source_system}] Appended to: {target_table} (partition: {run_date}, rows: {row_count})")
    print()


print("Setup complete.")

## Ingest Source 1 — HubSpot Leads

Lead capture and campus visit data from HubSpot CRM.
Contains all 10 students — the entry point of the pipeline.
Not all students will progress beyond this stage.

In [ ]:
ingest_to_bronze(
    filename      = "hubspot_leads.csv",
    source_system = "HUBSPOT",
    table_name    = "hubspot_leads"
)

## Ingest Source 2 — Dynamics 365 Applications

Application, offer, and decision data from the admissions CRM.
8 of the 10 students have applications — STU001 and STU002 dropped off
before applying and will have no rows here.

In [ ]:
ingest_to_bronze(
    filename      = "dynamics_applications.csv",
    source_system = "DYNAMICS",
    table_name    = "dynamics_applications"
)

## Ingest Source 3 — Paradigm SIS Enrolments

Confirmed enrolment records from the Student Information System.
6 students reach this stage — STU001 through STU004 never enrolled.
Contains fee, scholarship, credit point, and enrolment status data.

In [ ]:
ingest_to_bronze(
    filename      = "paradigm_enrolments.csv",
    source_system = "PARADIGM",
    table_name    = "paradigm_enrolments"
)

## Ingest Source 4 — Canvas LMS Attendance

Class session attendance records from the Learning Management System.
Each row is one student × one scheduled session.
`attended = 1` means the student attended. `attended = 0` means they were
scheduled but did not show up.

This raw data will be split into two Gold fact tables downstream:
- `fact_attendance_coverage` — all scheduled slots (`attended = 0 or 1`)
- `fact_attendance_actual` — only attended sessions (`attended = 1`)

In [ ]:
ingest_to_bronze(
    filename      = "canvas_attendance.csv",
    source_system = "CANVAS",
    table_name    = "canvas_attendance"
)

## Verification

Confirm all four Bronze tables exist and this run's partition has the correct row counts.

Expected for the initial load (`pipeline_run_date = 2024-01-15`):
- `bronze.hubspot_leads` — 10 rows
- `bronze.dynamics_applications` — 8 rows
- `bronze.paradigm_enrolments` — 6 rows
- `bronze.canvas_attendance` — 34 rows

On subsequent runs with updated CSVs, row counts will reflect the new file content.

In [ ]:
bronze_tables = [
    ("bronze.hubspot_leads",          "HUBSPOT"),
    ("bronze.dynamics_applications",  "DYNAMICS"),
    ("bronze.paradigm_enrolments",    "PARADIGM"),
    ("bronze.canvas_attendance",      "CANVAS"),
]

print(f"Partition being verified: _ingestion_date = {run_date}")
print()
print(f"{'Table':<42} {'This run':>10} {'All runs':>10}")
print("-" * 65)

for table, system in bronze_tables:
    df_all  = spark.table(table)
    df_run  = df_all.filter(
        (F.col("_source_system")  == system) &
        (F.col("_ingestion_date") == run_date)
    )
    print(f"{table:<42} {df_run.count():>10} {df_all.count():>10}")

print("-" * 65)
print("This run = rows in today's partition | All runs = cumulative total")

## Schema Inspection

Verify all four metadata columns are present on each table:
`_source_file`, `_source_system`, `_ingested_at`, `_ingestion_date`

`_ingestion_date` is the partition key — it must appear as the last column
before the partition folder structure is written to Delta.

In [ ]:
for table, _ in bronze_tables:
    print(f"\n--- {table} ---")
    spark.table(table).printSchema()

## Partition History

Show all ingestion dates loaded so far across all Bronze tables.
Each row here represents one pipeline run — the growing audit trail.

In [ ]:
for table, system in bronze_tables:
    print(f"\n--- {table} — partition history ---")
    (
        spark.table(table)
        .groupBy("_source_system", "_ingestion_date")
        .count()
        .orderBy("_ingestion_date")
        .show(truncate=False)
    )

## Summary

Bronze ingestion complete. All four source files are now appended as Delta table
partitions in the `bronze` schema under `_ingestion_date = {pipeline_run_date}`.

**What Silver will do with this:**
Silver reads only the partition matching `pipeline_run_date` — never the full table.
This means each pipeline run processes only new data, and MERGEs it into Silver.
Running the same date twice is safe — Silver's MERGE is idempotent.

**Next step:** Run `02_silver_transform.ipynb` with the same `pipeline_run_date`.

> **Rule:** Never delete or modify Bronze partitions.
> To correct bad data, fix the source CSV and re-run the pipeline with a new date.
> The bad partition stays as an audit record of what arrived.